In [1]:
import heapq
import math

In [2]:
class Node:
    """
    Represents a discrete search state in grid space.
    Priority in the min-heap is governed by total evaluation cost f = g + h.
    Tie-breaking prioritizes smaller heuristic distance h to drive exploration toward the goal.
    """
    def __init__(self, position, parent=None, g=0.0, h=0.0):
        self.position = position  # Coordinate tuple: (row, col)
        self.parent = parent      # Reference to predecessor Node instance
        self.g = g                # Cumulative path cost from start to current node
        self.h = h                # Heuristic estimate from current node to goal
        self.f = g + h            # Evaluation function priority

    def __lt__(self, other):
        if self.f == other.f:
            return self.h < other.h
        return self.f < other.f

    def __eq__(self, other):
        return self.position == other.position

    def __hash__(self):
        return hash(self.position)

In [3]:
def manhattan_distance(p1, p2):
    """Calculates L1 norm distance between two 2D grid coordinates."""
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

In [4]:
def base_astar_search(grid, start, goal):
    """
    Executes baseline 4-directional A* search on a grid matrix.
    0 = Walkable cell (Unit cost = 1.0)
    1 = Impassable obstacle
    """
    rows = len(grid)
    cols = len(grid[0])

    start_node = Node(start, None, g=0.0, h=manhattan_distance(start, goal))
    open_heap = []
    heapq.heappush(open_heap, start_node)

    open_dict = {start: start_node}
    closed_set = set()

    # 4-directional cardinal offsets: Up, Down, Left, Right
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    while open_heap:
        current_node = heapq.heappop(open_heap)

        if current_node.position in closed_set:
            continue

        closed_set.add(current_node.position)

        # Goal state evaluation
        if current_node.position == goal:
            path = []
            curr = current_node
            while curr:
                path.append(curr.position)
                curr = curr.parent
            return path[::-1], current_node.g

        r, c = current_node.position

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            neighbor_pos = (nr, nc)

            # Boundary check
            if 0 <= nr < rows and 0 <= nc < cols:
                # Obstacle check
                if grid[nr][nc] == 1:
                    continue
                if neighbor_pos in closed_set:
                    continue

                step_cost = 1.0  # Unit traversal cost
                tentative_g = current_node.g + step_cost

                # Check for cheaper existing path in open set
                if neighbor_pos in open_dict and tentative_g >= open_dict[neighbor_pos].g:
                    continue

                neighbor_node = Node(
                    position=neighbor_pos,
                    parent=current_node,
                    g=tentative_g,
                    h=manhattan_distance(neighbor_pos, goal)
                )

                open_dict[neighbor_pos] = neighbor_node
                heapq.heappush(open_heap, neighbor_node)

    return None, float('inf')